# Init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType
from pyspark.sql.functions import trim, col

# Reading from Bronze Table

In [0]:
df = spark.table('workspace.bronze.erp_loc_a101')

# Silver Transformation

## Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

In [0]:
df.limit(10).display()

## Customer ID Cleanup

In [0]:
df = df.withColumn('cid', F.regexp_replace(col("CID"), '-', ''))

## Country Normalisation

In [0]:
df = df.withColumn(
    'cntry',
    F.when(col("CNTRY") == 'DE','Germany')
    .when(col("CNTRY").isin('US', 'USA', 'Usa'), 'Unites States')
    .when((col("CNTRY") == '') | col("CNTRY").isNull(), 'N/A')
    .otherwise(col("CNTRY"))  
)

## Renaming Column

In [0]:
RENAME_MAP = {
    'cid' : 'customer_number',
    'cntry': 'country'
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

# Sanity Check 

In [0]:
df.limit(10).display()

# Writing it into Silver Table

In [0]:
%sql
drop table workspace.silver.erp_cutomer_location

In [0]:
df.write.mode('overwrite').format('delta').saveAsTable('workspace.silver.erp_cutomer_location')

In [0]:
%sql
select * from workspace.silver.erp_cutomer_location limit 10

In [0]:
%sql
select count(*), country
from workspace.silver.erp_cutomer_location
where country = 'N/A'
group by country
